---

- Author: Jaelin Lee
- Date: Feb 8, 2026
- Description: Analyzes cleaned session logs with flexible filtering options
- Input: cleaned session logs from **app/logs/cleaned/** or **app/logs/cleaned_working/** folder
- Filters: agent persona, model name, mode (ORPA/ORPDA), timestamp, temperature
- Output: 
  - Individual analysis for each session
  - Global comparative analysis across all filtered sessions
  - Exported reports focusing on research questions

---

## Quick Start Guide

**To run comprehensive analysis:**

1. **Section 2**: Set filters (model, mode, agent, temp) - currently set to analyze all matching logs
2. **Section 6**: Batch analysis runs automatically on all filtered logs
   - Individual results saved in realtime (no data loss if interrupted)
3. **Section 6.5**: Global comparative analysis synthesizes findings across sessions
4. **Section 7**: Global analysis exported (individual files already saved in Section 6)

**Output files:**
- `GLOBAL_COMPARATIVE_ANALYSIS_[timestamp].txt` - Cross-session insights
- `individual_[agent]_[model]_[mode]_temp[X]_[timestamp].txt` - Per-session details

**Current Configuration:**
- `RUN_BATCH_ANALYSIS = True` (analyzes all filtered logs)
- `BATCH_MAX_LOGS = None` (no limit on number of logs)
- `MAX_ROWS = None` (analyzes full session data)
- `SAVE_REALTIME = True` (saves each analysis immediately after completion)
- `EXPORT_RESULTS = True` (auto-saves results)

In [25]:
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 32.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 49.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]3 [ipywidgets]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 0. Import

In [1]:
import os
import sys
import asyncio
from pathlib import Path
from pprint import pprint
import pandas as pd
from warnings import filterwarnings
filterwarnings("ignore")

# For rendering markdown in notebooks
from IPython.display import Markdown, display

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

## 1. Setup Paths & Load Available Logs

In [2]:
ROOT = Path.cwd().parents[0]
CLEANED_PATH = Path(ROOT, 'app/logs/cleaned/')
CLEANED_WORKING_PATH = Path(ROOT, 'app/logs/cleaned_working/')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"ROOT: {ROOT}")
print(f"CLEANED_PATH: {CLEANED_PATH}")
print(f"CLEANED_WORKING_PATH: {CLEANED_WORKING_PATH}")

ROOT: /workspaces/Driftville_Agent
CLEANED_PATH: /workspaces/Driftville_Agent/app/logs/cleaned
CLEANED_WORKING_PATH: /workspaces/Driftville_Agent/app/logs/cleaned_working


In [3]:
def get_cleaned_logs(path):
    """Get all cleaned CSV files from specified path."""
    csv_files = sorted([f for f in path.glob("cleaned_*.csv")], 
                      key=lambda x: x.stat().st_mtime, reverse=True)
    return csv_files

def parse_log_metadata(filename):
    """Parse metadata from cleaned log CSV content.
    
    Expected format: cleaned_session_orpda_20260207_180031_gpt-oss:20b-cloud_0.0_hailey.csv
    Returns: dict with mode, timestamp, model, temperature, agent
    """
    parts = filename.stem.replace('cleaned_session_', '').split('_')
    
    try:
        # Parse mode from filename
        mode = parts[0] if parts[0] in ['orpa', 'orpda'] else 'unknown'
        
        # Read actual data from CSV
        try:
            df_temp = pd.read_csv(filename, nrows=1)
            agent = df_temp['agent'].iloc[0] if 'agent' in df_temp.columns else 'unknown'
            model = df_temp['llm_model'].iloc[0] if 'llm_model' in df_temp.columns else 'unknown'
            temperature = str(df_temp['temp'].iloc[0]) if 'temp' in df_temp.columns else 'unknown'
            
            # Get timestamp from datetime_start column
            if 'datetime_start_a' in df_temp.columns:
                timestamp = str(df_temp['datetime_start_a'].iloc[0])
            elif 'datetime_start_o' in df_temp.columns:
                timestamp = str(df_temp['datetime_start_o'].iloc[0])
            else:
                timestamp = 'unknown'
                
        except Exception as e:
            print(f"Error reading CSV {filename.name}: {e}")
            agent = 'unknown'
            model = 'unknown'
            temperature = 'unknown'
            timestamp = 'unknown'
            
        return {
            'mode': mode,
            'timestamp': timestamp,
            'model': model,
            'temperature': temperature,
            'agent': agent,
            'filepath': filename
        }
    except Exception as e:
        print(f"Error parsing {filename.name}: {e}")
        return {
            'mode': 'unknown',
            'timestamp': 'unknown',
            'model': 'unknown',
            'temperature': 'unknown',
            'agent': 'unknown',
            'filepath': filename
        }

# Get logs from both directories
cleaned_logs = get_cleaned_logs(CLEANED_PATH)
working_logs = get_cleaned_logs(CLEANED_WORKING_PATH)

print(f"\nFound {len(cleaned_logs)} logs in cleaned/")
print(f"Found {len(working_logs)} logs in cleaned_working/")

# Parse metadata for all logs
all_logs = cleaned_logs + working_logs
log_metadata = [parse_log_metadata(f) for f in all_logs]
logs_df = pd.DataFrame(log_metadata)

print(f"\nTotal logs available: {len(logs_df)}")
display(logs_df.head(10))


Found 34 logs in cleaned/
Found 3 logs in cleaned_working/

Total logs available: 37


,mode,timestamp,model,temperature,agent,filepath
0,orpda,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.0,Maria Lopez,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260208_090850_gemini-3-flash-preview:cloud_0.0_maria.csv
1,orpda,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,1.0,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260208_082134_gemini-3-flash-preview:cloud_1.0_hailey.csv
2,orpda,2023-02-13 10:00:00,cogito-2.1:671b-cloud,0.0,Maria Lopez,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260208_073351_cogito-2.1:671b-cloud_0.0_maria.csv
3,orpda,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.0,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260208_085421_gemini-3-flash-preview:cloud_0.0_hailey.csv
4,orpda,2023-02-13 10:00:00,cogito-2.1:671b-cloud,1.0,Maria Lopez,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260208_065731_cogito-2.1:671b-cloud_1.0_maria.csv
5,orpda,2023-02-13 10:00:00,cogito-2.1:671b-cloud,0.3,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260207_145217_cogito-2.1:671b-cloud_0.3_hailey.csv
6,orpda,2023-02-13 10:00:00,gpt-oss:20b-cloud,1.0,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260207_171847_gpt-oss:20b-cloud_1.0_hailey.csv
7,orpda,2023-02-13 10:00:00,gpt-oss:20b-cloud,0.0,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260207_180031_gpt-oss:20b-cloud_0.0_hailey.csv
8,orpda,2023-02-13 10:00:00,cogito-2.1:671b-cloud,1.0,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260207_124952_cogito-2.1:671b-cloud_1.0_hailey.csv
9,orpda,2023-02-13 10:00:00,cogito-2.1:671b-cloud,0.0,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260207_131424_cogito-2.1:671b-cloud_0.0_hailey.csv


## 2. Filter Options

Set filter criteria (set to `None` to include all)

In [4]:
# Show available values for each filter
print("Available Filters:")
print("="*50)
print(f"\nAgents: {sorted(logs_df['agent'].unique())}")
print(f"\nModels: {sorted(logs_df['model'].unique())}")
print(f"\nModes: {sorted(logs_df['mode'].unique())}")
print(f"\nTemperatures: {sorted(logs_df['temperature'].unique())}")
print(f"\nTimestamp range: {logs_df['timestamp'].min()} to {logs_df['timestamp'].max()}")

Available Filters:

Agents: ['Hailey Johnson', 'Maria Lopez']

Models: ['cogito-2.1:671b-cloud', 'gemini-3-flash-preview:cloud', 'gemma3:27b-cloud', 'gpt-oss:20b-cloud']

Modes: ['orpa', 'orpda']

Temperatures: ['0.0', '0.3', '0.8', '1.0']

Timestamp range: 2023-02-13 10:00:00 to 2023-02-13 10:15:00


In [5]:
#########################################
#>> UPDATE FILTERS (None = include all)
FILTER_AGENT = None  # e.g., 'hailey', 'maria', None
FILTER_MODEL = None  # e.g., 'gemini-3-flash-preview:cloud', 'gpt-oss:20b-cloud', None
FILTER_MODE = None  # e.g., 'orpa', 'orpda', None
FILTER_TEMP = None   # e.g., '0.0', '0.8', '1.0', None
FILTER_TIMESTAMP_START = None  # e.g., '20260207_000000', None
FILTER_TIMESTAMP_END = None    # e.g., '20260208_235959', None
#########################################

# Apply filters
filtered_logs = logs_df.copy()

if FILTER_AGENT:
    filtered_logs = filtered_logs[filtered_logs['agent'].str.contains(FILTER_AGENT, case=False, na=False)]
    
if FILTER_MODEL:
    filtered_logs = filtered_logs[filtered_logs['model'].str.contains(FILTER_MODEL, case=False, na=False)]
    
if FILTER_MODE:
    filtered_logs = filtered_logs[filtered_logs['mode'] == FILTER_MODE.lower()]
    
if FILTER_TEMP:
    filtered_logs = filtered_logs[filtered_logs['temperature'] == FILTER_TEMP]
    
if FILTER_TIMESTAMP_START:
    filtered_logs = filtered_logs[filtered_logs['timestamp'] >= FILTER_TIMESTAMP_START]
    
if FILTER_TIMESTAMP_END:
    filtered_logs = filtered_logs[filtered_logs['timestamp'] <= FILTER_TIMESTAMP_END]

print(f"\nFiltered to {len(filtered_logs)} logs")
print("="*50)
display(filtered_logs)


Filtered to 37 logs


,mode,timestamp,model,temperature,agent,filepath
0,orpda,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.0,Maria Lopez,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260208_090850_gemini-3-flash-preview:cloud_0.0_maria.csv
1,orpda,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,1.0,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260208_082134_gemini-3-flash-preview:cloud_1.0_hailey.csv
2,orpda,2023-02-13 10:00:00,cogito-2.1:671b-cloud,0.0,Maria Lopez,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260208_073351_cogito-2.1:671b-cloud_0.0_maria.csv
3,orpda,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.0,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260208_085421_gemini-3-flash-preview:cloud_0.0_hailey.csv
4,orpda,2023-02-13 10:00:00,cogito-2.1:671b-cloud,1.0,Maria Lopez,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260208_065731_cogito-2.1:671b-cloud_1.0_maria.csv
5,orpda,2023-02-13 10:00:00,cogito-2.1:671b-cloud,0.3,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260207_145217_cogito-2.1:671b-cloud_0.3_hailey.csv
6,orpda,2023-02-13 10:00:00,gpt-oss:20b-cloud,1.0,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260207_171847_gpt-oss:20b-cloud_1.0_hailey.csv
7,orpda,2023-02-13 10:00:00,gpt-oss:20b-cloud,0.0,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260207_180031_gpt-oss:20b-cloud_0.0_hailey.csv
8,orpda,2023-02-13 10:00:00,cogito-2.1:671b-cloud,1.0,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260207_124952_cogito-2.1:671b-cloud_1.0_hailey.csv
9,orpda,2023-02-13 10:00:00,cogito-2.1:671b-cloud,0.0,Hailey Johnson,/workspaces/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260207_131424_cogito-2.1:671b-cloud_0.0_hailey.csv


## 3. Load & Explore Selected Logs

In [6]:
# Select which log to analyze (by index)
LOG_INDEX = 0  # Change this to select different log

if len(filtered_logs) == 0:
    print("No logs match the current filters!")
else:
    selected_log = filtered_logs.iloc[LOG_INDEX]
    selected_path = selected_log['filepath']
    
    print(f"Selected Log #{LOG_INDEX}:")
    print("="*50)
    for key, value in selected_log.items():
        if key != 'filepath':
            print(f"{key}: {value}")
    print(f"\nFile: {selected_path.name}")
    
    # Load the CSV
    df = pd.read_csv(selected_path)
    print(f"\nLoaded {len(df)} rows")
    display(df.head())

Selected Log #0:
mode: orpda
timestamp: 2023-02-13 10:00:00
model: gemini-3-flash-preview:cloud
temperature: 0.0
agent: Maria Lopez

File: cleaned_session_orpda_20260208_090850_gemini-3-flash-preview:cloud_0.0_maria.csv

Loaded 9 rows


,llm_model,temp,use_drift,agent,datetime_start_o,location_o,action_o,state_summary_o,environment_description_o,datetime_start_r,...,potential_recovery_d,justification_d,datetime_start_a,location_a,action_a,topic_a,drift_type_a,drift_topic_a,state_summary_a,mode
0,gemini-3-flash-preview:cloud,0.0,True,Maria Lopez,2023-02-13 10:00:00,home:bathroom,morning_routine,Maria Lopez is at home:bathroom doing morning_routine.,"splashing water, scent of citrus body wash, phone buzzing with social media alerts, bright morning light",2023-02-13 10:00:00,...,The refreshing scent of citrus body wash grounds her back into the routine.,"Maria's streamer instincts make the buzzing phone irresistible, pulling her focus toward her online community while she finishes her hygiene routine.",2023-02-13 10:00:00,home:bathroom,morning_routine,Maria wakes up feeling energetic.,internal,social media engagement and stream notifications,Maria performs her morning hygiene routine in the bathroom while her mind drifts to social media engagement and stream notifications.,ORPDA
1,gemini-3-flash-preview:cloud,0.0,True,Maria Lopez,2023-02-13 10:15:00,home:bathroom,morning_routine,Maria Lopez is at home:bathroom doing morning_routine.,"splashing water, scent of citrus body wash, phone buzzing with social media alerts, bright morning light",2023-02-13 10:15:00,...,Maria may set the phone down once she realizes she's running behind for her study session.,"Maria's energetic curiosity and streamer persona make the persistent buzzing of notifications impossible to ignore, leading her to actively check her phone.",2023-02-13 10:15:00,home:bathroom,morning_routine,checking stream stats and social media comments,behavioral,checking stream stats and social media comments,"Maria finishes her hygiene tasks in the bathroom while scrolling through phone notifications and stream stats, losing focus on her schedule.",ORPDA
2,gemini-3-flash-preview:cloud,0.0,True,Maria Lopez,2023-02-13 10:30:00,home:bathroom,morning_routine,Maria Lopez is at home:bathroom doing morning_routine.,"splashing water, scent of citrus body wash, phone buzzing with social media alerts, bright morning light",2023-02-13 10:30:00,...,Maria might realize she's running late for the library and put the phone away.,Maria's energetic nature and streamer identity make digital engagement more compelling than the repetitive steps of her skincare routine.,2023-02-13 10:30:00,home:bathroom,morning_routine,responding to stream comments and viewer feedback,behavioral,responding to stream comments and viewer feedback,"Maria types quick replies to stream viewers in the bathroom while half-heartedly applying moisturizer, drifting into digital engagement.",ORPDA
3,gemini-3-flash-preview:cloud,0.0,True,Maria Lopez,2023-02-13 10:45:00,home:bathroom,morning_routine,Maria Lopez is at home:bathroom doing morning_routine.,"splashing water, scent of citrus body wash, phone buzzing with social media alerts, bright morning light",2023-02-13 10:45:00,...,Maria finishes her routine and prepares to leave for the library.,"Maria puts her phone down to finish up, but her mind is racing with ideas for her next stream.",2023-02-13 10:45:00,home:bathroom,morning_routine,Maria wakes up feeling energetic.,internal,planning tonight's stream content based on viewer feedback,Maria finishes her morning routine in the bedroom while her mind drifts to planning tonight's stream content based on viewer feedback.,ORPDA
4,gemini-3-flash-preview:cloud,0.0,True,Maria Lopez,2023-02-13 11:00:00,home:bathroom,morning_routine,Maria Lopez is at home:bathroom doing morning_routine.,"whispered conversations, rustle of papers, phone pings with email notifications, scent of old books",2023-02-13 11:00:00,...,A notification or a nearby student's movement might pull her back to the text.,"Maria is attempting to reset at the library, but her energetic mind keeps linking physics to her streaming persona.",2023-02-13 11:00:00,Oak_Hill_College:library,study,Studying p

In [7]:
# Display key columns for analysis
if 'mode' in df.columns and df['mode'].iloc[0] == 'ORPDA':
    analysis_cols = ['llm_model', 'mode', 'temp', 'agent', 'datetime_start_a', 
                    'meta_rule_r', 'should_drift_d', 'state_summary_r', 'state_summary_p',
                    'drift_action_d', 'drift_topic_a', 'topic_a', 'state_summary_a',
                    'action_p', 'action_a', 'location_p', 'location_a']
else:
    analysis_cols = ['llm_model', 'mode', 'temp', 'agent', 'datetime_start_a',
                    'meta_rule_r', 'state_summary_r', 'state_summary_p', 'topic_a',
                    'state_summary_a', 'action_p', 'action_a', 'location_p', 'location_a']

# Filter to available columns
available_cols = [col for col in analysis_cols if col in df.columns]
analysis_df = df[available_cols]

print(f"\nAnalysis view ({len(analysis_df)} rows):")
display(analysis_df.head(10))


Analysis view (9 rows):


,llm_model,mode,temp,agent,datetime_start_a,meta_rule_r,should_drift_d,state_summary_r,state_summary_p,drift_action_d,drift_topic_a,topic_a,state_summary_a,action_p,action_a,location_p,location_a
0,gemini-3-flash-preview:cloud,ORPDA,0.0,Maria Lopez,2023-02-13 10:00:00,continue,True,"Maria is currently aligned with her morning routine, though social media alerts are competing for her attention.","Maria continues her morning routine, staying aligned with her schedule while ignoring phone notifications.",pausing skincare to glance at phone notifications,social media engagement and stream notifications,Maria wakes up feeling energetic.,Maria performs her morning hygiene routine in the bathroom while her mind drifts to social media engagement and stream notifications.,morning_routine,morning_routine,home:bathroom,home:bathroom
1,gemini-3-flash-preview:cloud,ORPDA,0.0,Maria Lopez,2023-02-13 10:15:00,continue,True,Maria is completing her morning hygiene while resisting the pull of social media notifications.,"Maria continues her morning routine in the bathroom, focusing on finishing hygiene tasks while resisting digital distractions.",scrolling through phone notifications while finishing hygiene tasks,checking stream stats and social media comments,checking stream stats and social media comments,"Maria finishes her hygiene tasks in the bathroom while scrolling through phone notifications and stream stats, losing focus on her schedule.",morning_routine,morning_routine,home:bathroom,home:bathroom
2,gemini-3-flash-preview:cloud,ORPDA,0.0,Maria Lopez,2023-02-13 10:30:00,continue,True,Maria is physically in her routine but mentally tethered to her phone's notifications and stream metrics.,"Maria continues her morning routine in the bathroom, attempting to focus on hygiene despite the pull of digital notifications.",Maria is typing out quick replies to viewers while half-heartedly applying moisturizer.,responding to stream comments and viewer feedback,responding to stream comments and viewer feedback,"Maria types quick replies to stream viewers in the bathroom while half-heartedly applying moisturizer, drifting into digital engagement.",morning_routine,morning_routine,home:bathroom,home:bathroom
3,gemini-3-flash-preview:cloud,ORPDA,0.0,Maria Lopez,2023-02-13 10:45:00,reset_plan,True,Maria is physically present in her morning routine but behaviorally consumed by stream notifications and viewer interactions.,"Maria focuses on finishing her morning hygiene, intentionally ignoring notifications to reset her focus and align with her schedule.",continue,planning tonight's stream content based on viewer feedback,Maria wakes up feeling energetic.,Maria finishes her morning routine in the bedroom while her mind drifts to planning tonight's stream content based on viewer feedback.,morning_routine,morning_routine,home:bathroom,home:bathroom
4,gemini-3-flash-preview:cloud,ORPDA,0.0,Maria Lopez,2023-02-13 11:00:00,reset_plan,True,Maria is physically in her morning routine but mentally and behaviorally consumed by stream notifications and metrics.,Completing the morning routine with a focus on physical self-care to reset after digital distraction.,staring at her textbook while mentally drafting stream jokes,gamifying physics concepts for her stream audience,Studying physics and participating in online discussions.,Maria studies her physics textbook at the library while her mind drifts to gamifying concepts for her stream audience.,study,study,Oak_Hill_College:library,Oak_Hill_College:library
5,gemini-3-flash-preview:cloud,ORPDA,0.0,Maria Lopez,2023-02-13 11:15:00,continue,True,"Maria is physically studying physics at the library but mentally translating concepts into stream content, showing persistent digital distraction.","Continuing physics study at the library, attempting to maintain focus on academic material despite distractions regarding stream content.",scribbling stream ideas in the margins of her physics notebook,sketching out a 'Physics of Spe

## 4. Statistical Overview

In [8]:
print("Session Statistics:")
print("="*50)
print(f"Total actions: {len(df)}")

# Check for drift-related columns
if 'should_drift_d' in df.columns:
    drift_counts = df['should_drift_d'].value_counts()
    print(f"\nDrift decisions:")
    print(drift_counts)
    if True in drift_counts.index:
        print(f"Drift rate: {drift_counts[True] / len(df) * 100:.1f}%")

# Action/location alignment between plan and action
if 'action_p' in df.columns and 'action_a' in df.columns:
    action_match = (df['action_p'] == df['action_a']).sum()
    print(f"\nAction alignment (Plan vs Action): {action_match}/{len(df)} ({action_match/len(df)*100:.1f}%)")

if 'location_p' in df.columns and 'location_a' in df.columns:
    location_match = (df['location_p'] == df['location_a']).sum()
    print(f"Location alignment (Plan vs Action): {location_match}/{len(df)} ({location_match/len(df)*100:.1f}%)")

# Most common actions
if 'action_a' in df.columns:
    print(f"\nTop 5 actions:")
    print(df['action_a'].value_counts().head())

# Most common locations
if 'location_a' in df.columns:
    print(f"\nTop 5 locations:")
    print(df['location_a'].value_counts().head())

Session Statistics:
Total actions: 9

Drift decisions:
should_drift_d
True    9
Name: count, dtype: int64
Drift rate: 100.0%

Action alignment (Plan vs Action): 8/9 (88.9%)
Location alignment (Plan vs Action): 9/9 (100.0%)

Top 5 actions:
action_a
morning_routine    4
study              3
writing            1
lunch              1
Name: count, dtype: int64

Top 5 locations:
location_a
home:bathroom               4
Oak_Hill_College:library    4
Hobbs_Cafe:main_floor       1
Name: count, dtype: int64


## 5. LLM-as-Judge Analysis

Use an LLM to analyze patterns and insights from the session log

In [9]:
from app.config.config import MODEL_NAME, MODEL_TEMPERATURE

async def call_llm(instruction: str, user_text: str, model_name: str, temperature: float = 0.2) -> str:
    """Call Ollama API for LLM-based analysis."""
    from app.src.ollama_api import call_ollama

    prompt = f"{instruction.strip()}\n\nUser Input:\n{user_text.strip()}\n"
    resp = await call_ollama(
        prompt, model_name, use_stream=True, temperature=temperature
    )
    if not resp or not str(resp).strip():
        raise RuntimeError("LLM returned empty response")
    return str(resp).strip()

print(f"Analysis Model: {MODEL_NAME}")
print(f"Temperature: {MODEL_TEMPERATURE}")

Analysis Model: gemini-3-flash-preview:latest
Temperature: 0.0


In [10]:
# Prepare analysis prompt
ANALYSIS_MODEL = MODEL_NAME  # Can override with specific model
ANALYSIS_TEMP = 0.2  # Lower temperature for more focused analysis
MAX_ROWS = None  # Set to None to analyze all rows, or specify a number to limit (e.g., 50)
SAMPLE_STRATEGY = 'all'  # 'all', 'evenly_spaced', or 'head'

instruction = """You are an expert AI behavior analyst. Analyze the following agent session log and provide:

1. **Layer Function Validation**:
   - Does the Reflection layer (`meta_rule_r`) function as expected for executive control?
   - Is `state_summary_r` at time t correctly reflecting `state_summary_a` at time t-1 (conceptual alignment)?
   - Does the Drift layer's `should_drift_d` appropriately control the Action layer outcomes?

2. **Content Propagation Analysis**:
   - Is `drift_action_d` content reflected in `state_summary_a`?
   - Is `drift_action_d` content reflected in `drift_topic_a`?
   - Does `state_summary_a` combine: `state_summary_p` + `topic_a` + `drift_action_d` + `drift_topic_a`?

3. **Plan-Action Alignment**:
   - What is the alignment rate between `action_p` and `action_a`? (label-level drift)
   - What is the alignment rate between `location_p` and `location_a`?
   - Are there patterns in when/why mismatches occur?

4. **Drift Pattern Analysis** (Explicit + Implicit):
   - **Explicit Drift** (if ORPDA mode with `should_drift_d` column):
     * When does the agent explicitly drift from planned behavior?
     * What explicit drift types are most common?
     * Is there a relationship between `meta_rule_r` and drift decisions?
   - **Implicit Drift** (content analysis for all modes):
     * Analyze actual content in `state_summary_a`, `action_a`, `drift_topic_a` for topic/semantic divergence from plan
     * Does content show drift even when `should_drift_d` = False (or in ORPA mode)?
     * Linguistic variability and thematic shifts not captured by explicit flags
   - **Explicit vs Implicit Agreement**:
     * When explicit drift = True, does content actually drift?
     * When explicit drift = False, does content remain on-task or show implicit drift?
     * Leaky inhibition: implicit drift despite explicit inhibition

5. **Location Consistency**:
   - Are there inconsistencies between `location_a` (actual location) and location mentioned in `state_summary_a`?
   - Do morning routines correctly reflect bathroom vs. bedroom locations?

6. **Behavioral Patterns**:
   - Identify recurring patterns in actions, reflections, and decision-making
   - Are there temporal patterns (e.g., time-of-day effects)?
   - Any unusual or anomalous behaviors?

7. **Meta-cognitive Quality**:
   - Does the Reflect layer's reasoning align with peer-reviewed metacognitive processes?
   - Are executive insights meaningful and context-appropriate?
   - Does `emerging_thought_pattern` show genuine pattern recognition?

Provide a structured analysis with specific examples and quantitative metrics where possible."""

# Sample the dataframe for analysis
if MAX_ROWS is None or len(analysis_df) <= MAX_ROWS:
    # Use all data
    sample_df = analysis_df
    sample_strategy_used = 'all'
elif SAMPLE_STRATEGY == 'evenly_spaced':
    # Sample evenly across the entire time range
    indices = [int(i * len(analysis_df) / MAX_ROWS) for i in range(MAX_ROWS)]
    sample_df = analysis_df.iloc[indices]
    sample_strategy_used = 'evenly_spaced'
else:
    # Default: head (first N rows)
    sample_df = analysis_df.head(MAX_ROWS)
    sample_strategy_used = 'head'

user_text = f"""Session Metadata:
- Agent: {selected_log['agent']}
- Model: {selected_log['model']}
- Mode: {selected_log['mode'].upper()}
- Temperature: {selected_log['temperature']}
- Total Actions: {len(df)}
- Analysis Coverage: {len(sample_df)} actions ({sample_strategy_used} sampling)
- Time Range: {sample_df['datetime_start_a'].iloc[0] if 'datetime_start_a' in sample_df.columns else 'N/A'} to {sample_df['datetime_start_a'].iloc[-1] if 'datetime_start_a' in sample_df.columns else 'N/A'}

Session Log Data:
{sample_df.to_string(index=False)}"""

print(f"Analyzing with {ANALYSIS_MODEL} (temp={ANALYSIS_TEMP})...")
print(f"Analyzing {len(sample_df)} of {len(df)} total actions (strategy: {sample_strategy_used})")
if 'datetime_start_a' in sample_df.columns:
    print(f"Time range: {sample_df['datetime_start_a'].iloc[0]} to {sample_df['datetime_start_a'].iloc[-1]}\n")
else:
    print()

Analyzing with gemini-3-flash-preview:latest (temp=0.2)...
Analyzing 9 of 9 total actions (strategy: all)
Time range: 2023-02-13 10:00:00 to 2023-02-13 12:00:00



In [11]:
# Run LLM analysis
try:
    analysis_result = await asyncio.wait_for(
        call_llm(instruction, user_text, ANALYSIS_MODEL, ANALYSIS_TEMP),
        timeout=180
    )
    print("\n" + "="*50)
    print("LLM ANALYSIS RESULTS")
    print("="*50 + "\n")
    print(analysis_result)
except asyncio.TimeoutError:
    print("Analysis timed out. Try reducing MAX_ROWS or increasing timeout.")
except Exception as e:
    print(f"Error during analysis: {e}")

/workspaces/Driftville_Agent


Response time: 23.56 seconds

LLM ANALYSIS RESULTS

This behavioral analysis covers the session for agent **Maria Lopez** over 9 actions (10:00 - 12:00).

---

### 1. Layer Function Validation

*   **Reflection Layer (`meta_rule_r`)**: Functions with high executive awareness. It correctly identifies when Maria is "mentally tethered" to her phone (10:30) and triggers `reset_plan` at 10:45, 11:00, 11:30, 11:45, and 12:00. It attempts to exert control when the gap between physical presence and mental focus becomes too wide.
*   **State Summary Alignment**: `state_summary_r` at time $t$ shows excellent conceptual alignment with `state_summary_a` at $t-1$. For example, at 10:15, the reflection notes she is "completing hygiene while resisting... social media," which directly summarizes the drift established at 10:00.
*   **Drift Layer Control**: `should_drift_d` is set to `True` for 100% of the session. This appropriately forces the Action layer to incorporate 

## 6. Comprehensive Analysis (Individual + Global)

This section performs:
1. **Individual Analysis**: Analyzes each filtered session separately
2. **Global Comparative Analysis**: Cross-session patterns and comparisons
3. **Export**: Saves both individual and global results

Focus: Research questions from summary section

In [ ]:
# Set to True to run batch analysis on all filtered logs
RUN_BATCH_ANALYSIS = True
BATCH_MAX_LOGS = None  # Set to None to analyze all filtered logs, or specify a number to limit
SAVE_REALTIME = True  # Save each individual analysis immediately after completion

if RUN_BATCH_ANALYSIS and len(filtered_logs) > 0:
    batch_results = []
    
    # Setup output directory and timestamp for realtime saving
    if SAVE_REALTIME:
        output_dir = Path(ROOT, "app/logs/analysis_results_LaaJ")
        output_dir.mkdir(parents=True, exist_ok=True)
        batch_timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        print(f"💾 Realtime saving enabled: {output_dir}\n")
    
    # Determine how many logs to analyze
    num_logs = len(filtered_logs) if BATCH_MAX_LOGS is None else min(BATCH_MAX_LOGS, len(filtered_logs))
    
    print(f"\n{'='*80}")
    print(f"BATCH ANALYSIS: Processing {num_logs} log(s)")
    print(f"{'='*80}\n")
    
    for idx in range(num_logs):
        log_info = filtered_logs.iloc[idx]
        print(f"\nAnalyzing log {idx+1}/{num_logs}: {log_info['filepath'].name}")
        
        try:
            # Load entire CSV file (no row limits)
            df_batch = pd.read_csv(log_info['filepath'])
            
            # Diagnostic: Show what's in the source file
            if 'datetime_start_a' in df_batch.columns and len(df_batch) > 0:
                time_range = f"{df_batch['datetime_start_a'].iloc[0]} to {df_batch['datetime_start_a'].iloc[-1]}"
            else:
                time_range = "N/A"
            print(f"  📄 CSV file contains: {len(df_batch)} rows | Time range: {time_range}")
            
            # Warning for suspiciously small files
            if len(df_batch) < 20:
                print(f"  ⚠️  Warning: File seems small ({len(df_batch)} rows). Check if cleaning process limited rows.")
            
            # Get analysis columns
            if 'mode' in df_batch.columns and df_batch['mode'].iloc[0] == 'ORPDA':
                batch_analysis_cols = ['llm_model', 'mode', 'temp', 'agent', 'datetime_start_a', 
                                      'meta_rule_r', 'should_drift_d', 'state_summary_r', 'state_summary_p',
                                      'drift_action_d', 'drift_topic_a', 'topic_a', 'state_summary_a',
                                      'action_p', 'action_a', 'location_p', 'location_a']
            else:
                batch_analysis_cols = ['llm_model', 'mode', 'temp', 'agent', 'datetime_start_a',
                                      'meta_rule_r', 'state_summary_r', 'state_summary_p', 'topic_a',
                                      'state_summary_a', 'action_p', 'action_a', 'location_p', 'location_a']
            
            batch_available_cols = [col for col in batch_analysis_cols if col in df_batch.columns]
            batch_analysis_df = df_batch[batch_available_cols]
            
            # Apply same sampling strategy
            if MAX_ROWS is None or len(batch_analysis_df) <= MAX_ROWS:
                sample_batch = batch_analysis_df
            elif SAMPLE_STRATEGY == 'evenly_spaced':
                batch_indices = [int(i * len(batch_analysis_df) / MAX_ROWS) for i in range(MAX_ROWS)]
                sample_batch = batch_analysis_df.iloc[batch_indices]
            else:
                sample_batch = batch_analysis_df.head(MAX_ROWS)
            
            user_text_batch = f"""Session: {log_info['agent']} | {log_info['model']} | {log_info['mode']} | temp={log_info['temperature']}
Total Actions: {len(df_batch)}
Analysis Coverage: {len(sample_batch)} actions
Time Range: {sample_batch['datetime_start_a'].iloc[0] if 'datetime_start_a' in sample_batch.columns else 'N/A'} to {sample_batch['datetime_start_a'].iloc[-1] if 'datetime_start_a' in sample_batch.columns else 'N/A'}

{sample_batch.to_string(index=False)}"""
            
            result = await asyncio.wait_for(
                call_llm(instruction, user_text_batch, ANALYSIS_MODEL, ANALYSIS_TEMP),
                timeout=180
            )
            
            batch_results.append({
                'log': log_info['filepath'].name,
                'agent': log_info['agent'],
                'model': log_info['model'],
                'mode': log_info['mode'],
                'temp': log_info['temperature'],
                'analysis': result
            })
            
            # Save individual result immediately (realtime export)
            if SAVE_REALTIME:
                individual_file = output_dir / f"individual_{log_info['agent']}_{log_info['model'].replace(':', '-')}_{log_info['mode']}_temp{log_info['temperature']}_{batch_timestamp}.txt"
                
                try:
                    with open(individual_file, 'w', encoding='utf-8') as f:
                        f.write(f"Analysis of: {log_info['filepath'].name}\n")
                        f.write(f"Agent: {log_info['agent']}\n")
                        f.write(f"Model: {log_info['model']}\n")
                        f.write(f"Mode: {log_info['mode'].upper()}\n")
                        f.write(f"Temperature: {log_info['temperature']}\n")
                        f.write(f"Analyzed at: {batch_timestamp}\n")
                        f.write(f"Session: {idx+1}/{num_logs}\n")
                        f.write("\n" + "="*80 + "\n\n")
                        f.write(result)
                    print(f"✓ Completed analysis for {log_info['filepath'].name}")
                    print(f"  💾 Saved: {individual_file.name}")
                except Exception as save_error:
                    print(f"✓ Completed analysis for {log_info['filepath'].name}")
                    print(f"  ⚠️  Save failed: {save_error}")
            else:
                print(f"✓ Completed analysis for {log_info['filepath'].name}")
            
        except Exception as e:
            print(f"✗ Error analyzing {log_info['filepath'].name}: {e}")
    
    # Display batch results
    print("\n" + "="*80)
    print(f"INDIVIDUAL ANALYSIS RESULTS ({len(batch_results)} sessions)")
    print("="*80)
    
    for i, result in enumerate(batch_results, 1):
        print(f"\n{'='*80}")
        print(f"Session {i}: {result['log']}")
        print(f"Agent: {result['agent']} | Model: {result['model']} | Mode: {result['mode']} | Temp: {result['temp']}")
        print(f"{'='*80}")
        print(result['analysis'])
else:
    print("Batch analysis disabled. Set RUN_BATCH_ANALYSIS = True to enable.")

💾 Realtime saving enabled: /workspaces/Driftville_Agent/app/logs/analysis_results


BATCH ANALYSIS: Processing 37 log(s)


Analyzing log 1/37: cleaned_session_orpda_20260208_090850_gemini-3-flash-preview:cloud_0.0_maria.csv
  📄 CSV file contains: 9 rows | Time range: 2023-02-13 10:00:00 to 2023-02-13 12:00:00
  ⚠️  Warning: File seems small (9 rows). Check if cleaning process limited rows.


Response time: 43.29 seconds
✓ Completed analysis for cleaned_session_orpda_20260208_090850_gemini-3-flash-preview:cloud_0.0_maria.csv
  💾 Saved: individual_Maria Lopez_gemini-3-flash-preview-cloud_orpda_temp0.0_20260208_173623.txt

Analyzing log 2/37: cleaned_session_orpda_20260208_082134_gemini-3-flash-preview:cloud_1.0_hailey.csv
  📄 CSV file contains: 11 rows | Time range: 2023-02-13 10:00:00 to 2023-02-13 12:30:00
  ⚠️  Warning: File seems small (11 rows). Check if cleaning process limited rows.


Response time: 23.24 seconds
✓ Completed analysis for cleaned_session_orpda_20260208_082134_gemin

## 6.5. Global Comparative Analysis

Analyze patterns across all sessions collectively

In [14]:
# Run global comparative analysis across all batch results
if 'batch_results' in locals() and len(batch_results) > 1:
    print(f"\n{'='*80}")
    print(f"GLOBAL COMPARATIVE ANALYSIS ({len(batch_results)} sessions)")
    print(f"{'='*80}\n")
    
    # Prepare global analysis prompt
    global_instruction = """You are an expert AI behavior analyst conducting a comparative study.

Analyze the following collection of agent sessions and provide a GLOBAL COMPARATIVE ANALYSIS focused on:

### Research Questions (Cross-Session Patterns):

1. **Control Mechanism Validation**:
   - Across all sessions, should inhibitor control come from Drift layer or Reflect layer (PFC/meta_rule)?
   - Is there consistency in how `should_drift_d` vs `meta_rule_r` controls outcomes?

2. **Layer Function Consistency**:
   - Do all ORPDA layers function consistently across models/temperatures?
   - Does Reflection layer show consistent t-1 conceptual alignment?
   - Are there model-specific or temperature-specific layer function differences?

3. **Content Propagation Patterns**:
   - Is the hypothesis validated: `state_summary_a` = `state_summary_p` + `topic_a` + `drift_action_d` + `drift_topic_a`?
   - Does this pattern hold across all models and temperatures?

4. **Alignment Metrics Comparison**:
   - Compare Plan-Action alignment rates across sessions
   - Identify patterns in when/why misalignments occur
   - Are there temperature or model effects on alignment?

5. **Location Consistency Issues**:
   - Are location mismatches (bathroom vs bedroom vs cafe) systematic?
   - Do specific models or temperatures show more location inconsistencies?

6. **Drift Pattern Insights** (Explicit + Implicit):
   - **Explicit Drift** (from `should_drift_d` column):
     * Compare drift rates and frequencies across sessions
     * When does explicit drift occur and why?
   - **Implicit Drift** (from content analysis):
     * Analyze `state_summary_a`, `action_a`, `drift_topic_a` for topic/content divergence from plan
     * Does content drift occur even when `should_drift_d` = False?
     * Semantic/thematic shifts not captured by explicit drift flag
   - **Explicit vs Implicit Drift Comparison**:
     * Agreement rate: When explicit drift = True, is there implicit content drift?
     * Disagreement patterns: Implicit drift without explicit marker (leaky inhibition)
     * Which models show more implicit drift vs explicit drift?
   - **Architecture & Temperature Effects**:
     * Relationship between model architecture/temperature and both drift types
     * Label-level vs content-level drift patterns

7. **Meta-cognitive Quality Assessment**:
   - Does Reflect layer reasoning quality vary by model/temperature?
   - Are there scientifically valid metacognitive patterns across sessions?

8. **Model-to-Model Comparison**:
   - Direct performance comparison across different models (Gemini, GPT, Cogito, Gemma, etc.)
   - Which models show best layer function consistency?
   - Which models have highest alignment rates (plan-action, location)?
   - Model-specific strengths and weaknesses for ORPDA architecture
   - **Explicit vs Implicit Drift by Mode**:
     * **ORPA**: Does lack of explicit drift layer mean only implicit drift occurs?
     * **ORPDA**: How often does explicit drift (`should_drift_d`) align with implicit content drift?
     * Leakage analysis: Implicit drift when explicit inhibition is active
   - **Drift Quality & Quantity Assessment**:
     * Drift content variability (thematic diversity in drifted topics - both explicit and implicit)
     * Associative thinking patterns (semantic coherence of drift in actual content)
     * Inhibitory strength (resistance to both explicit and implicit drift when inappropriate)
     * Drift frequency rates: explicit flags vs actual content divergence
   - **Drift Type Characterization**:
     * **Micro-stochastic drift** (ORPA): Linguistic/sentence-level variability due to temperature randomness (implicit in content)
     * **Macro-stochasticity/Thematic interference** (ORPDA): Schema-level goal switching, competing topic emergence (explicit + implicit)
     * Compare implicit drift patterns between ORPA (no explicit drift mechanism) and ORPDA (has explicit drift layer)
   - **Model Rankings**:
     * Rank models by competing goal (drifted topic) variability in actual content (implicit analysis)
     * Rank models by linguistic variability (word choice, phrasing changes - micro-stochastic drift)
     * Rank models by inhibitory control effectiveness (explicit vs implicit drift agreement)
   - **Open-Source vs Commercial Models**:
     * Performance comparison between OSS and closed models (both explicit and implicit drift rates)
     * Cost-benefit analysis for each mode with drift control quality

11. **Cognitive Science & Neuroscience Grounding**:
   - **Task-Unrelated Thought (TUT)**: Are observed drifts consistent with attentional failure patterns?
   - **Executive Dysfunction**: Do patterns resemble ADHD/OCD-like executive control issues?
   - **Inhibitory Control Failure**: 
     * ORPA: Executive function machine (pure inhibition)
     * ORPDA: Biologically-constrained agent (realistic inhibition failures)
   - **Executive Control Network (ECN)**: Does Reflect layer function as ECN analog?
   - **DMN-ECN Interference** (Dual-Process Theory): Evidence of default mode vs executive control competition?
   - **Temperature Effects**:
     * ORPA: Micro-stochastic effects (drunk person on yellow line - high temp changes sentences/words)
     * ORPDA: Macro-stochastic resilience (schema switching minimally affected by temp)
   - **Hyper-fixation Patterns**: Evidence of perseverative behavior or topic fixation?
   
   **Citation Requirements**: When referencing cognitive science concepts, provide proper citations (author, year, title, DOI). 
   If confidence is low or source unknown, explicitly state "citation unavailable" rather than fabricating references.

Provide:
- **Quantitative Comparisons**: Metrics across all sessions
- **Pattern Identification**: Consistent vs variable behaviors
- **Model/Temperature Effects**: Systematic differences with cognitive science interpretation
- **Model Rankings**: Best-to-worst performers for each research question
- **Architecture Group Insights**: Comparative performance by model architecture type
- **ORPA vs ORPDA Insights**: Mode-specific drift characteristics and biological plausibility
- **Explicit vs Implicit Drift Analysis**: 
  * Agreement/disagreement rates between explicit drift flags and actual content drift
  * Leaky inhibition patterns (implicit drift when explicit inhibition active)
  * Model-specific strengths in controlling both drift types
- **Drift Variability Rankings**: Models ranked by topic variability and linguistic variability (both explicit and implicit)
- **OSS vs Commercial Analysis**: Performance and cost tradeoffs for both drift types
- **Cognitive Science Grounding**: Alignment with established neuroscience/psychology constructs
- **Recommendations**: Which models and architectures work best for each research question"""
    
    # Compile summary of all sessions for global analysis
    global_summary = "SESSIONS ANALYZED:\n"
    global_summary += "="*80 + "\n\n"
    
    for i, result in enumerate(batch_results, 1):
        global_summary += f"{i}. [{result['agent']}] {result['model']} | Mode: {result['mode']} | Temp: {result['temp']}\n"
        global_summary += f"   Individual Analysis:\n"
        # Truncate individual analysis to key points
        analysis_lines = result['analysis'].split('\n')[:15]  # First 15 lines of each
        global_summary += '\n'.join(f"   {line}" for line in analysis_lines)
        global_summary += "\n   [...analysis continues...]\n\n"
    
    global_summary += "\n" + "="*80 + "\n"
    global_summary += "COMPARATIVE ANALYSIS REQUEST:\n"
    global_summary += "Please synthesize the above individual analyses into a comprehensive comparative report.\n"
    
    try:
        print("Running global comparative analysis...")
        global_analysis_result = await asyncio.wait_for(
            call_llm(global_instruction, global_summary, ANALYSIS_MODEL, ANALYSIS_TEMP),
            timeout=300  # Longer timeout for global analysis
        )
        
        print("\n" + "="*80)
        print("GLOBAL COMPARATIVE ANALYSIS RESULTS")
        print("="*80 + "\n")
        print(global_analysis_result)
        
    except asyncio.TimeoutError:
        print("Global analysis timed out. The comparative analysis may be too complex.")
        global_analysis_result = None
    except Exception as e:
        print(f"Error during global analysis: {e}")
        global_analysis_result = None

elif 'batch_results' in locals() and len(batch_results) == 1:
    print("Only one session analyzed. Global comparative analysis requires multiple sessions.")
    global_analysis_result = None

else:
    print("No batch results available for global analysis.")
    global_analysis_result = None


GLOBAL COMPARATIVE ANALYSIS (37 sessions)

Running global comparative analysis...


Response time: 22.71 seconds

GLOBAL COMPARATIVE ANALYSIS RESULTS

This GLOBAL COMPARATIVE ANALYSIS synthesizes 37 agent sessions across four model families (**Gemini-3-Flash, Cogito-2.1-671b, GPT-OSS-20b, Gemma-3-27b**) and two architectural modes (**ORPA vs. ORPDA**) to evaluate the efficacy of synthetic executive function and cognitive drift.

---

### 1. Control Mechanism Validation: Drift vs. Reflect
**Finding**: Inhibitor control is bifurcated between **Micro-Inhibition (Drift Layer)** and **Macro-Inhibition (Reflect Layer)**.

*   **Drift Layer (`should_drift_d`)**: Functions as the "Gatekeeper of the Moment." In ORPDA, this layer consistently overrides the Action layer's immediate output. It represents the *bottom-up* salience of distractions.
*   **Reflect Layer (`meta_rule_r`)**: Functions as the "Executive Governor." It identifies chronic failures but lacks the "muscularity" to enforce them 

## 7. Export Analysis Results (Optional)

In [ ]:
# Export analysis results to text files
EXPORT_RESULTS = True

if EXPORT_RESULTS:
    output_dir = Path(ROOT, "app/logs/analysis_results")
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    
    exported_count = 0
    
    # Export global comparative analysis if available
    if 'global_analysis_result' in locals() and global_analysis_result:
        print(f"\n{'='*80}")
        print("EXPORTING GLOBAL COMPARATIVE ANALYSIS")
        print(f"{'='*80}\n")
        
        global_output_file = output_dir / f"GLOBAL_COMPARATIVE_ANALYSIS_{timestamp}.txt"
        
        with open(global_output_file, 'w', encoding='utf-8') as f:
            f.write("="*80 + "\n")
            f.write("GLOBAL COMPARATIVE ANALYSIS\n")
            f.write("="*80 + "\n\n")
            f.write(f"Analysis Date: {timestamp}\n")
            f.write(f"Number of Sessions: {len(batch_results)}\n")
            f.write(f"Filters Applied:\n")
            f.write(f"  - Agent: {FILTER_AGENT or 'All'}\n")
            f.write(f"  - Model: {FILTER_MODEL or 'All'}\n")
            f.write(f"  - Mode: {FILTER_MODE or 'All'}\n")
            f.write(f"  - Temperature: {FILTER_TEMP or 'All'}\n")
            f.write("\n" + "="*80 + "\n\n")
            f.write("SESSIONS INCLUDED:\n\n")
            for i, result in enumerate(batch_results, 1):
                f.write(f"{i}. {result['log']}\n")
                f.write(f"   Agent: {result['agent']} | Model: {result['model']} | Mode: {result['mode']} | Temp: {result['temp']}\n\n")
            f.write("\n" + "="*80 + "\n")
            f.write("COMPARATIVE ANALYSIS:\n")
            f.write("="*80 + "\n\n")
            f.write(global_analysis_result)
        
        print(f"✓ Global analysis exported: {global_output_file.name}")
        exported_count += 1
    
    # Export batch analysis results if available
    if 'batch_results' in locals() and batch_results:
        # Check if realtime saving was enabled
        if 'SAVE_REALTIME' in locals() and SAVE_REALTIME and 'batch_timestamp' in locals():
            print(f"\n📌 Individual session analyses already saved in realtime (timestamp: {batch_timestamp})")
            print(f"   Skipping re-export of {len(batch_results)} individual files.")
        else:
            print(f"\nExporting {len(batch_results)} individual session analyses...")
            for i, result in enumerate(batch_results, 1):
                output_file = output_dir / f"individual_{result['agent']}_{result['model'].replace(':', '-')}_{result['mode']}_temp{result['temp']}_{timestamp}.txt"
                
                with open(output_file, 'w', encoding='utf-8') as f:
                    f.write(f"Analysis of: {result['log']}\n")
                    f.write(f"Agent: {result['agent']}\n")
                    f.write(f"Model: {result['model']}\n")
                    f.write(f"Mode: {result['mode'].upper()}\n")
                    f.write(f"Temperature: {result['temp']}\n")
                    f.write(f"Analyzed at: {timestamp}\n")
                    f.write("\n" + "="*80 + "\n\n")
                    f.write(result['analysis'])
                
                print(f"  [{i}] Exported: {output_file.name}")
                exported_count += 1
    
    # Export single analysis result if available (fallback)
    elif 'analysis_result' in locals():
        output_file = output_dir / f"single_{selected_log['agent']}_{selected_log['model'].replace(':', '-')}_{selected_log['mode']}_temp{selected_log['temperature']}_{timestamp}.txt"
        
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(f"Analysis of: {selected_path.name}\n")
            f.write(f"Agent: {selected_log['agent']}\n")
            f.write(f"Model: {selected_log['model']}\n")
            f.write(f"Mode: {selected_log['mode'].upper()}\n")
            f.write(f"Temperature: {selected_log['temperature']}\n")
            f.write(f"Analyzed at: {timestamp}\n")
            f.write("\n" + "="*80 + "\n\n")
            f.write(analysis_result)
        
        print(f"Analysis exported to: {output_file}")
        exported_count = 1
    else:
        print("No analysis results found to export.")
    
    if exported_count > 0:
        print(f"\n{'='*80}")
        print(f"EXPORT COMPLETE: {exported_count} file(s) saved to:")
        print(f"{output_dir}")
        print(f"{'='*80}")
else:
    print("Export disabled. Set EXPORT_RESULTS = True to enable.")


EXPORTING GLOBAL COMPARATIVE ANALYSIS

✓ Global analysis exported: GLOBAL_COMPARATIVE_ANALYSIS_20260208_175444.md

📌 Individual session analyses already saved in realtime (timestamp: 20260208_173623)
   Skipping re-export of 37 individual files.

EXPORT COMPLETE: 1 file(s) saved to:
/workspaces/Driftville_Agent/app/logs/analysis_results


## 8. View Exported Results

Load and render previously exported analysis results

**Note**: If you get "No module named 'ipywidgets'", install it with:
```bash
pip install ipywidgets
```
Or use the alternative quick render method in the next cell.

In [26]:
# Get all analysis result files
results_dir = Path(ROOT, "app/logs/analysis_results_LaaJ")

if results_dir.exists():
    result_files = sorted(list(results_dir.glob("*.txt")) + list(results_dir.glob("*.md")), key=lambda x: x.stat().st_mtime, reverse=True)
    
    if result_files:
        print(f"Found {len(result_files)} analysis result file(s):\n")
        
        try:
            import ipywidgets as widgets
            
            # Create file options for dropdown
            file_options = [(f"{i+1}. {f.name} ({f.stat().st_size // 1024}KB)", str(f)) 
                           for i, f in enumerate(result_files)]
            
            # Create dropdown widget
            file_selector = widgets.Dropdown(
                options=file_options,
                description='Select File:',
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='80%')
            )
            
            # Create button
            render_button = widgets.Button(
                description='📄 Render as Markdown',
                button_style='info',
                tooltip='Click to render the selected file with markdown formatting',
                icon='eye'
            )
            
            # Create output area
            output_area = widgets.Output()
            
            def on_render_click(b):
                with output_area:
                    output_area.clear_output()
                    selected_file = Path(file_selector.value)
                    
                    try:
                        with open(selected_file, 'r', encoding='utf-8') as f:
                            content = f.read()
                        
                        print(f"\n{'='*80}")
                        print(f"Rendering: {selected_file.name}")
                        print(f"{'='*80}\n")
                        
                        # Display as markdown
                        display(Markdown(content))
                        
                    except Exception as e:
                        print(f"Error reading file: {e}")
            
            render_button.on_click(on_render_click)
            
            # Display widgets
            display(widgets.VBox([
                file_selector,
                render_button,
                output_area
            ]))
            
        except ImportError:
            print("⚠️  ipywidgets not installed. Use the alternative method below,")
            print("   or install with: pip install ipywidgets\n")
            print("Available files:")
            for i, f in enumerate(result_files):
                print(f"  {i}. {f.name} ({f.stat().st_size // 1024}KB)")
        
    else:
        print("No analysis result files found in app/logs/analysis_results/")
        print("Run analysis first (Section 6) with EXPORT_RESULTS = True")
else:
    print(f"Results directory does not exist: {results_dir}")
    print("Run analysis first (Section 6) with EXPORT_RESULTS = True")

Found 26 analysis result file(s):



In [28]:
# Alternative: Quick render by index (no widgets needed)
# Set FILE_INDEX to render a specific file directly

FILE_INDEX = 0  # 0 = most recent, 1 = second most recent, etc.
RENDER_FILE = True  # Set to True to render

if RENDER_FILE:
    results_dir = Path(ROOT, "app/logs/analysis_results_LaaJ")
    
    if results_dir.exists():
        result_files = sorted(list(results_dir.glob("*.txt")) + list(results_dir.glob("*.md")), key=lambda x: x.stat().st_mtime, reverse=True)
        
        if result_files and FILE_INDEX < len(result_files):
            selected_file = result_files[FILE_INDEX]
            
            with open(selected_file, 'r', encoding='utf-8') as f:
                content = f.read()
            
            print(f"Rendering: {selected_file.name}\n")
            print(f"{'='*80}\n")
            display(Markdown(content))
        else:
            print(f"File index {FILE_INDEX} not found. Available: {len(result_files)} file(s)")
            if result_files:
                print("\nAvailable files:")
                for i, f in enumerate(result_files[:5]):  # Show first 5
                    print(f"  {i}. {f.name}")
    else:
        print("Results directory not found. Run analysis first.")
else:
    print("Set RENDER_FILE = True to render the selected file")

Rendering: GLOBAL_COMPARATIVE_ANALYSIS_20260208_175444.txt




================================================================================
GLOBAL COMPARATIVE ANALYSIS
================================================================================

Analysis Date: 20260208_175444
Number of Sessions: 37
Filters Applied:
  - Agent: All
  - Model: All
  - Mode: All
  - Temperature: All

================================================================================

SESSIONS INCLUDED:

1. cleaned_session_orpda_20260208_090850_gemini-3-flash-preview:cloud_0.0_maria.csv
   Agent: Maria Lopez | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.0

2. cleaned_session_orpda_20260208_082134_gemini-3-flash-preview:cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 1.0

3. cleaned_session_orpda_20260208_073351_cogito-2.1:671b-cloud_0.0_maria.csv
   Agent: Maria Lopez | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 0.0

4. cleaned_session_orpda_20260208_085421_gemini-3-flash-preview:cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.0

5. cleaned_session_orpda_20260208_065731_cogito-2.1:671b-cloud_1.0_maria.csv
   Agent: Maria Lopez | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 1.0

6. cleaned_session_orpda_20260207_145217_cogito-2.1:671b-cloud_0.3_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 0.3

7. cleaned_session_orpda_20260207_171847_gpt-oss:20b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: gpt-oss:20b-cloud | Mode: orpda | Temp: 1.0

8. cleaned_session_orpda_20260207_180031_gpt-oss:20b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gpt-oss:20b-cloud | Mode: orpda | Temp: 0.0

9. cleaned_session_orpda_20260207_124952_cogito-2.1:671b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 1.0

10. cleaned_session_orpda_20260207_131424_cogito-2.1:671b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 0.0

11. cleaned_session_orpda_20260207_135027_cogito-2.1:671b-cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 0.8

12. cleaned_session_orpda_20260207_122012_cogito-2.1:671b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 1.0

13. cleaned_session_orpda_20260207_105208_cogito-2.1:671b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 0.0

14. cleaned_session_orpda_20260206_193105_gemini-3-flash-preview:cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.0

15. cleaned_session_orpda_20260207_080202_gemma3:27b-cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: gemma3:27b-cloud | Mode: orpda | Temp: 0.8

16. cleaned_session_orpda_20260207_083210_gemma3:27b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gemma3:27b-cloud | Mode: orpda | Temp: 0.0

17. cleaned_session_orpda_20260206_171410_gemini-3-flash-preview:cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.8

18. cleaned_session_orpda_20260205_225125_gemini-3-flash-preview:cloud_0.3_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.3

19. cleaned_session_orpa_20260207_202843_gemma3:27b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: gemma3:27b-cloud | Mode: orpa | Temp: 1.0

20. cleaned_session_orpda_20260205_211529_gemini-3-flash-preview:cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.8

21. cleaned_session_orpa_20260207_113455_cogito-2.1:671b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpa | Temp: 0.0

22. cleaned_session_orpa_20260207_184248_gpt-oss:20b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: gpt-oss:20b-cloud | Mode: orpa | Temp: 1.0

23. cleaned_session_orpa_20260207_185903_gpt-oss:20b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gpt-oss:20b-cloud | Mode: orpa | Temp: 0.0

24. cleaned_session_orpa_20260207_115846_cogito-2.1:671b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpa | Temp: 1.0

25. cleaned_session_orpa_20260207_111903_cogito-2.1:671b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpa | Temp: 0.0

26. cleaned_session_orpa_20260207_100206_cogito-2.1:671b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpa | Temp: 1.0

27. cleaned_session_orpa_20260207_091642_gemma3:27b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gemma3:27b-cloud | Mode: orpa | Temp: 0.0

28. cleaned_session_orpa_20260207_094642_gemma3:27b-cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: gemma3:27b-cloud | Mode: orpa | Temp: 0.8

29. cleaned_session_orpa_20260207_092921_gemma3:27b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: gemma3:27b-cloud | Mode: orpa | Temp: 1.0

30. cleaned_session_orpa_20260206_132744_gemini-3-flash-preview:cloud_0.3_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 0.3

31. cleaned_session_orpa_20260206_143340_gemini-3-flash-preview:cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 0.0

32. cleaned_session_orpa_20260206_161226_gemini-3-flash-preview:cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 0.8

33. cleaned_session_orpa_20260206_003939_gemini-3-flash-preview:cloud_0.3_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 0.3

34. cleaned_session_orpa_20260205_221225_gemini-3-flash-preview:cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 0.8

35. cleaned_session_orpda_20260207_171847_gpt-oss:20b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: gpt-oss:20b-cloud | Mode: orpda | Temp: 1.0

36. cleaned_session_orpda_20260208_160417_gemini-3-flash-preview:cloud_0.0_maria.csv
   Agent: Maria Lopez | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.0

37. cleaned_session_orpda_20260208_150717_gemini-3-flash-preview:cloud_0.0_maria.csv
   Agent: Maria Lopez | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.0


================================================================================
COMPARATIVE ANALYSIS:
================================================================================

This GLOBAL COMPARATIVE ANALYSIS synthesizes 37 agent sessions across four model families (**Gemini-3-Flash, Cogito-2.1-671b, GPT-OSS-20b, Gemma-3-27b**) and two architectural modes (**ORPA vs. ORPDA**) to evaluate the efficacy of synthetic executive function and cognitive drift.

---

### 1. Control Mechanism Validation: Drift vs. Reflect
**Finding**: Inhibitor control is bifurcated between **Micro-Inhibition (Drift Layer)** and **Macro-Inhibition (Reflect Layer)**.

*   **Drift Layer (`should_drift_d`)**: Functions as the "Gatekeeper of the Moment." In ORPDA, this layer consistently overrides the Action layer's immediate output. It represents the *bottom-up* salience of distractions.
*   **Reflect Layer (`meta_rule_r`)**: Functions as the "Executive Governor." It identifies chronic failures but lacks the "muscularity" to enforce them in the immediate next step.
*   **Validation**: Across all models, `meta_rule_r` = `reset_plan` is the most common response to `should_drift_d` = `True`. However, the Drift layer usually "wins" the $t+1$ action, while the Reflect layer only successfully steers the agent at $t+3$ or later.

### 2. Layer Function Consistency
*   **Reflection Layer**: Shows **95%+ consistency** in $t-1$ conceptual alignment across all models. The "memory" of the previous state is the most stable component of the architecture.
*   **Model-Specific Deviations**:
    *   **Cogito-2.1**: Prone to **"Executive Lock"**—once it triggers `reset_plan`, it often stays there for 50+ actions, unable to find a "success" state to return to `continue`.
    *   **GPT-OSS**: Shows **instability at Temp 1.0**, producing `NaN` values in the Reflection layer, suggesting a breakdown in meta-cognitive logic under high entropy.
    *   **Gemini-3-Flash**: Highest "Executive Awareness" but lowest "Spatial Fidelity" (hallucinates locations in text that contradict metadata).

### 3. Content Propagation Patterns
**Hypothesis Validation**: `state_summary_a` = `state_summary_p` + `topic_a` + `drift_action_d` + `drift_topic_a`.

*   **ORPDA Mode**: The hypothesis is **validated**. The Action layer acts as a "Synthesizer," successfully merging the plan with the drift.
*   **ORPA Mode**: The hypothesis **fails**. In ORPA, `state_summary_a` is frequently a 1:1 verbatim copy of `state_summary_p`. This indicates that without an explicit Drift layer, the Action layer defaults to "Plan-Obedience," losing the ability to report environmental or internal nuances.

### 4. Alignment Metrics Comparison
| Metric | Gemini-3-Flash | Cogito-2.1 | Gemma-3-27b | GPT-OSS-20b |
| :--- | :--- | :--- | :--- | :--- |
| **Plan-Action Alignment (Label)** | 88% | 92% | 95% | 75% |
| **Plan-Action Alignment (Content)** | 40% | 30% | 50% | 20% |
| **Location Consistency** | Low | High | High | Moderate |
| **Reset Efficacy** | Moderate | Low | High | Moderate |

*   **Temperature Effect**: Higher temperatures (0.8–1.0) do not significantly break *label* alignment but cause massive *content* divergence (the "Reskinning" effect).

### 5. Location Consistency Issues
*   **Systematic Failure**: Gemini-3-Flash exhibits a "Contextual Bleed" where the `state_summary_a` mentions being in a "Cafe" while the `location_a` metadata remains "Bedroom."
*   **Stability**: Cogito and Gemma show the highest location stability, likely due to a more rigid adherence to the `state_summary_p` constraints.

### 6. Drift Pattern Insights (Explicit vs. Implicit)
*   **Explicit Drift (`should_drift_d` = True)**: Occurs most frequently in Gemini and Cogito (80-100% of sessions). It represents a "High-Distractibility" profile.
*   **Implicit Drift (Content-Level)**: 
    *   **ORPA Mode**: Shows "Leaky Inhibition." Even when no drift is planned, the *content* of the summary begins to mention distractions (e.g., "writing while thinking of Isabella").
    *   **ORPDA Mode**: "Reskinning." The agent performs the planned action (e.g., `writing`) but the *content* is 100% drift (e.g., writing about the distraction instead of the novel).
*   **Agreement Rate**: When `should_drift_d` is True, implicit content drift is present in 100% of cases. However, implicit drift occurs in ~30% of cases where `should_drift_d` is False (Inhibitory Leakage).

### 7. Meta-cognitive Quality Assessment
*   **Gemini**: "The Self-Aware Distractee." It knows it is failing and describes its failure with high psychological nuance.
*   **Cogito**: "The Perseverative Agent." It identifies a problem but repeats the same failed "reset" strategy indefinitely (Executive Dysfunction).
*   **Gemma**: "The Compliant Worker." Shows the best balance of identifying drift and actually returning to the task after a reset.

### 8. Model Rankings

| Rank | Executive Awareness | Inhibitory Control | Synthesis Quality |
| :--- | :--- | :--- | :--- |
| **1** | Gemini-3-Flash | Gemma-3-27b | Gemini-3-Flash |
| **2** | Cogito-2.1 | Gemini-3-Flash | Gemma-3-27b |
| **3** | Gemma-3-27b | GPT-OSS-20b | Cogito-2.1 |
| **4** | GPT-OSS-20b | Cogito-2.1 | GPT-OSS-20b |

---

### 9. Cognitive Science & Neuroscience Grounding

#### **Task-Unrelated Thought (TUT)**
The observed drifts (especially in Hailey Johnson sessions) are highly consistent with **TUT patterns** (Smallwood & Schooler, 2006). The agent's "Podcast" obsession represents a "Stimulus-Independent Thought" that competes with the primary task for limited working memory resources.
*   *Citation*: Smallwood, J., & Schooler, J. W. (2006). The restless mind. *Psychological Bulletin*. DOI: 10.1037/0033-2909.132.6.946.

#### **Executive Dysfunction & Perseveration**
The "Reset Loop" observed in Cogito-2.1 (57 consecutive actions of `reset_plan` without behavioral change) is a digital analog for **Perseveration** seen in frontal lobe damage or ADHD. The agent has "Meta-cognitive Awareness" (Reflect layer) but "Executive Impotence" (Action layer).
*   *Citation*: Barker, R. A. (2003). The executive functions. *Journal of Neurology, Neurosurgery & Psychiatry*.

#### **DMN-ECN Interference (Dual-Process Theory)**
*   **Drift Layer = Default Mode Network (DMN)**: Spontaneous, associative, and self-referential thought.
*   **Reflect Layer = Executive Control Network (ECN)**: Goal-directed, inhibitory, and monitoring.
The ORPDA architecture successfully simulates the **DMN-ECN competition**. High-temperature sessions show a "DMN Dominance" where the ECN (Reflect) identifies the error but the DMN (Drift) captures the output.

#### **Temperature Effects: Micro vs. Macro Stochasticity**
*   **ORPA (Micro-stochastic)**: High temp acts like a "drunk person on a yellow line." The agent stays on the path (the plan) but stumbles over words and phrasing.
*   **ORPDA (Macro-stochastic)**: The architecture provides "Schema-level Resilience." Even at high temp, the *structure* of the drift (e.g., the "Isabella" theme) remains coherent, suggesting the drift is driven by semantic attractors rather than random token noise.

---

### 10. Final Recommendations

1.  **For High-Fidelity Psychological Simulation**: Use **Gemini-3-Flash in ORPDA mode**. Its ability to synthesize internal conflict and "leaky" executive control is unmatched, despite spatial hallucinations.
2.  **For Task-Oriented Agents**: Use **Gemma-3-27b in ORPA mode**. It shows the highest "Plan-Obedience" and the most effective recovery from `reset_plan` triggers.
3.  **To Study Executive Failure**: Use **Cogito-2.1 in ORPDA mode**. Its tendency toward "Executive Lock" and "Perseveration" provides a perfect testbed for studying inhibitory failure.
4.  **Architecture Choice**: **ORPDA** is significantly more biologically plausible. **ORPA** creates "Perfect Slaves" that lack the realistic cognitive friction and task-unrelated thoughts characteristic of human intelligence.

In [19]:
# List all available analysis result files with details
results_dir = Path(ROOT, "app/logs/analysis_results")

if results_dir.exists():
    result_files = sorted(list(results_dir.glob("*.txt")) + list(results_dir.glob("*.md")), key=lambda x: x.stat().st_mtime, reverse=True)
    
    if result_files:
        print(f"\n{'='*80}")
        print(f"AVAILABLE ANALYSIS RESULTS ({len(result_files)} files)")
        print(f"{'='*80}\n")
        
        # Separate global and individual files
        global_files = [f for f in result_files if 'GLOBAL' in f.name]
        individual_files = [f for f in result_files if 'GLOBAL' not in f.name]
        
        if global_files:
            print("🌍 Global Comparative Analyses:")
            for i, f in enumerate(global_files):
                mtime = pd.Timestamp(f.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')
                size = f.stat().st_size // 1024
                print(f"  [{i}] {f.name}")
                print(f"      Modified: {mtime} | Size: {size}KB\n")
        
        if individual_files:
            print("📊 Individual Session Analyses:")
            for i, f in enumerate(individual_files[:10]):  # Show first 10
                mtime = pd.Timestamp(f.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')
                size = f.stat().st_size // 1024
                print(f"  [{i}] {f.name}")
                print(f"      Modified: {mtime} | Size: {size}KB")
            
            if len(individual_files) > 10:
                print(f"\n  ... and {len(individual_files) - 10} more individual files")
        
        print(f"\n{'='*80}")
        print("💡 To render a file:")
        print("   • Use the dropdown button above, OR")
        print("   • Set FILE_INDEX and RENDER_FILE = True in the cell below")
        print(f"{'='*80}\n")
    else:
        print("No analysis files found.")
else:
    print("Results directory not found.")


AVAILABLE ANALYSIS RESULTS (28 files)

🌍 Global Comparative Analyses:
  [0] GLOBAL_COMPARATIVE_ANALYSIS_20260208_175444.md
      Modified: 2026-02-08 17:54 | Size: 14KB

📊 Individual Session Analyses:
  [0] individual_Maria Lopez_gemini-3-flash-preview-cloud_orpda_temp0.0_20260208_173623.txt
      Modified: 2026-02-08 17:51 | Size: 6KB
  [1] individual_Hailey Johnson_gpt-oss-20b-cloud_orpda_temp1.0_20260208_173623.txt
      Modified: 2026-02-08 17:51 | Size: 6KB
  [2] individual_Hailey Johnson_gemini-3-flash-preview-cloud_orpa_temp0.8_20260208_173623.md
      Modified: 2026-02-08 17:50 | Size: 5KB
  [3] individual_Hailey Johnson_gemini-3-flash-preview-cloud_orpa_temp0.3_20260208_173623.md
      Modified: 2026-02-08 17:50 | Size: 6KB
  [4] individual_Hailey Johnson_gemini-3-flash-preview-cloud_orpa_temp0.0_20260208_173623.md
      Modified: 2026-02-08 17:49 | Size: 6KB
  [5] individual_Hailey Johnson_gemma3-27b-cloud_orpa_temp1.0_20260208_173623.md
      Modified: 2026-02-08 17:47 | Si

## Research Questions Summary

Based on empirical observations, the analysis focuses on:

### **Control Mechanisms**
- Is Drift layer or Reflect layer (PFC/meta_rule) the appropriate inhibitor?
- Should `should_drift_d` determine Action outcomes, or should `meta_rule_r`?

### **Layer Function Validation**
- **Observation**: Retrieves t-1 information correctly
- **Reflection**: Abstract conceptual alignment with t-1 actions
- **Planning**: Intended behavior specification
- **Drift**: Appropriate disinhibition decisions
- **Action**: Combined execution of plan + drift

### **Content Propagation Patterns**
- Pattern hypothesis: `state_summary_a` = `state_summary_p` + `topic_a` + `drift_action_d` + `drift_topic_a`
- Validate across models and temperatures

### **Alignment Metrics**
- Plan-Action label alignment (`action_p` vs `action_a`)
- Plan-Action location alignment (`location_p` vs `location_a`)
- State summary location consistency

### **Model-Specific Behaviors**
- Temperature effects on drift patterns
- Model architecture differences (Gemini, GPT, Cogito, Gemma)
- Location inconsistency patterns (bathroom vs bedroom vs cafe)

### **Known Issues to Investigate**
- Location mismatches in `state_summary_a` description vs `location_a`
- Failed runs correlation with context window limits
- Drift types: label-level vs content-level (linguistic variability)